# Library Imports

In [ ]:
# Time Copilot
!pip install timecopilot

# Data prep & all of them
!pip install utilsforecast
import pandas as pd
import matplotlib.pyplot as plt
from utilsforecast.plotting import plot_series
from utilsforecast.losses import bias, rmse, mae, mape, bias
from utilsforecast.evaluation import evaluate



In [ ]:
# TimeCopilot Imports
from timecopilot import TimeCopilotForecaster
from timecopilot.models.foundation.chronos import Chronos
from timecopilot.models.foundation.moirai import Moirai
from timecopilot.models.foundation.timesfm import TimesFM

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.


# Data Preparation (this is shortened from first version)

In [ ]:
df_base = pd.read_parquet('/content/sample_hotels-1.parquet')
df_base.head()

,unique_id,ds,holiday_flag,target_day,target_month,target_year,location_type,hotel_type,y,otb_1,...,otb_51,otb_52,otb_53,otb_54,otb_55,otb_56,otb_57,otb_58,otb_59,otb_60
1430,hotel_0,2022-01-01,no,Sat,Jan,2022,NonSuburban,Resorts & Destinations,0.975309,0.679012,...,0.197531,0.197531,0.197531,0.185185,0.160494,0.160494,0.160494,0.160494,0.160494,0.160494
1431,hotel_0,2022-01-02,no,Sun,Jan,2022,NonSuburban,Resorts & Destinations,0.493827,0.308642,...,0.074074,0.074074,0.074074,0.074074,0.074074,0.061728,0.061728,0.061728,0.061728,0.049383
1432,hotel_0,2022-01-03,no,Mon,Jan,2022,NonSuburban,Resorts & Destinations,0.456790,0.358025,...,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691
1433,hotel_0,2022-01-04,no,Tue,Jan,2022,NonSuburban,Resorts & Destinations,0.592593,0.419753,...,0.074074,0.074074,0.074074,0.074074,0.061728,0.061728,0.037037,0.037037,0.024691,0.024691
1434,hotel_0,2022-01-05,no,Wed,Jan,2022,NonSuburban,Resorts & Destinations,0.530864,0.407407,...,0.074074,0.074074,0.074074,0.074074,0.074074,0.049383,0.049383,0.024691,0.024691,0.012346


Drop unimformative variable

In [ ]:
df_base = df_base.drop(columns=['target_year'])

Dummies

In [ ]:
 # Convert holiday flag to boolean manually
 df_base['holiday_flag'] = df_base['holiday_flag'].astype('bool')

In [ ]:
cat_cols = [
    'target_day',
    'target_month',
    'location_type',
    'hotel_type'
]

for col in cat_cols:
    df_base[col] = df_base[col].astype('category')

In [ ]:
df_base = pd.get_dummies(df_base, columns=cat_cols, drop_first=True)

OTB

In [ ]:
# Drop all OTB values before otb_28 because that information wouldn't actually be available at the time of forecasting
# By only keeping otb_28 and beyond, I ensure the model doesn't "cheat" by looking at data from inside the 28-day period
columns_to_drop = [f'otb_{i}' for i in range(1, 28)]
df_base = df_base.drop(columns=columns_to_drop)
display(df_base.head())

,unique_id,ds,y,otb_28,otb_29,otb_30,otb_31,otb_32,otb_33,otb_34,...,target_month_Jun,target_month_Mar,target_month_May,target_month_Nov,target_month_Oct,target_month_Sep,location_type_NonSuburban,hotel_type_Key Central Business District,hotel_type_Other High Leisure Mix,hotel_type_Resorts & Destinations
1430,hotel_0,2022-01-01,0.975309,0.296296,0.283951,0.296296,0.296296,0.283951,0.259259,0.234568,...,False,False,False,False,False,False,True,False,False,True
1431,hotel_0,2022-01-02,0.493827,0.086420,0.086420,0.086420,0.086420,0.086420,0.086420,0.086420,...,False,False,False,False,False,False,True,False,False,True
1432,hotel_0,2022-01-03,0.456790,0.061728,0.061728,0.061728,0.061728,0.061728,0.061728,0.049383,...,False,False,False,False,False,False,True,False,False,True
1433,hotel_0,2022-01-04,0.592593,0.111111,0.111111,0.111111,0.111111,0.098765,0.098765,0.098765,...,False,False,False,False,False,False,True,False,False,True
1434,hotel_0,2022-01-05,0.530864,0.111111,0.111111,0.111111,0.111111,0.111111,0.111111,0.111111,...,False,False,False,False,False,False,True,False,False,True


Drop Hotels

In [ ]:
hotels_to_drop = ['hotel_28', 'hotel_77']
df_base = df_base[df_base['unique_id'].isin(hotels_to_drop) == False]

Test/Train and Pred/No Pred Split

In [ ]:
# cutoff
cutoff = '2023-05-31'

#train/test split
train = df_base[df_base['ds'] <= cutoff]
test = df_base[df_base['ds'] > cutoff]

# full df no pred
df_no_pred = df_base[['unique_id', 'ds', 'y']]

# train
train_base = train[['unique_id', 'ds', 'y']] # Nixtila format dataset
train_ml = train.copy() # full dataset for ML

test_base = test[['unique_id', 'ds', 'y']]
test_ml = test.copy()

# TimeCopilot Cross Validation & Evaluation

In [ ]:
tcf = TimeCopilotForecaster(
    models = [
        Chronos(repo_id="amazon/chronos-bolt-small"),
        Moirai(),
        TimesFM(repo_id="google/timesfm-2.5-200m-pytorch", alias="TimesFM-2.5"),
    ]
)

In [ ]:
t_cv = tcf.cross_validation(
    df = df_no_pred,
    h = 28, freq = 'D',
    n_windows = 5,
    step_size = 28
)

0it [00:00, ?it/s]

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/191M [00:00<?, ?B/s]


100%|██████████| 2/2 [00:00<00:00,  3.08it/s]
1it [00:05,  5.35s/it]
100%|██████████| 2/2 [00:00<00:00,  3.57it/s]
2it [00:06,  2.73s/it]
100%|██████████| 2/2 [00:00<00:00,  2.35it/s]
3it [00:07,  2.05s/it]
100%|██████████| 2/2 [00:00<00:00,  2.22it/s]
4it [00:08,  1.79s/it]
100%|██████████| 2/2 [00:00<00:00,  3.22it/s]
5it [00:09,  1.97s/it]
0it [00:00, ?it/s]

...

model.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]


0it [00:00, ?it/s]
17it [05:07, 18.11s/it]
1it [05:22, 322.65s/it]
0it [00:00, ?it/s]
17it [05:05, 17.94s/it]
2it [10:30, 314.12s/it]
0it [00:00, ?it/s]
17it [05:11, 18.34s/it]
3it [15:45, 314.25s/it]
0it [00:00, ?it/s]
17it [05:07, 18.09s/it]
4it [20:57, 313.30s/it]
0it [00:00, ?it/s]
17it [05:12, 18.39s/it]
5it [26:12, 314.50s/it]
0it [00:00, ?it/s]

config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/925M [00:00<?, ?B/s]


100%|██████████| 1/1 [00:26<00:00, 26.44s/it]
1it [00:48, 48.31s/it]
100%|██████████| 1/1 [00:26<00:00, 26.28s/it]
2it [01:15, 35.66s/it]
100%|██████████| 1/1 [00:26<00:00, 26.20s/it]
3it [01:41, 31.58s/it]
100%|██████████| 1/1 [00:26<00:00, 26.98s/it]
4it [02:09, 29.95s/it]
100%|██████████| 1/1 [00:28<00:00, 28.40s/it]
5it [02:38, 31.64s/it]


In [ ]:
eval_tcf = evaluate(
    df = t_cv,
    metrics = [mape, rmse, mae, bias],
    models = ['Chronos', 'Moirai', 'TimesFM-2.5']
)

In [ ]:
# Drop identifiers and average metrics across all series and cutoffs for easier metrics comparison
eval_tcf = eval_tcf.drop(columns=['cutoff']).groupby(['unique_id', 'metric']).mean().reset_index(inplace=False)
eval_tcf

# Save evaluation as parquete
eval_tcf.to_parquet('eval_tcf_1.parquet', index=False)